# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform basic analysis on the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

## Dataset Source
The dataset Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant's DatasetMetadata object

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review the available record sets, their fields, and collect the `@id`s for later use. All references use the `@id` property as specified by the Croissant schema.

We'll inspect the structure for record sets, fields, and columns.

In [ ]:
# List record set (@id)s and show their fields and columns
record_sets = metadata.record_sets
print(f"{len(record_sets)} record set(s) found:\n")
for i, rs in enumerate(record_sets):
    print(f"{i+1}) Record Set name: {rs.name}\n   @id: {rs.id}\n   Description: {rs.description}")
    print("   Fields:")
    for field in rs.fields:
        print(f"     - {field.name} (@id: {field.id}) | dataType: {getattr(field, 'data_type', None)}")
    print("   Columns:")
    for col in rs.columns:
        print(f"     - {col.name} (@id: {col.id}) | source: {col.source}")
    print("")

### View a sample record from the first record set

All record set and field accesses below reference their `@id` directly.

In [ ]:
# For demonstration, show a sample record from the first record set
if record_sets:
    sample_rs = record_sets[0]
    sample_rs_id = sample_rs.id
    print(f"First record set @id: {sample_rs_id}\n")
    print("Sample records:")
    for i, rec in enumerate(dataset.records(record_set=sample_rs_id)):
        print(json.dumps(rec, indent=2))
        if i >= 1:
            break
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction

Let's extract each record set into a pandas DataFrame for further analysis.

**Note:** All `@id`s are used for programmatic access as recommended by the Croissant schema and FAIR best practices.

In [ ]:
all_dataframes = {}
rs_ids = [rs.id for rs in record_sets]

# Load data for each record set
for rs_id in rs_ids:
    print(f"Loading records for record set {rs_id}...")
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        df = pd.DataFrame(recs)
        all_dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample data (first 5 records):\n{df.head()}\n")
    else:
        print("  No records found for this set.\n")

## 4. Exploratory Data Analysis (EDA)

Apply basic data filtering, normalization, and grouping for a numeric field from a chosen record set.

### Steps:
1. Choose a numeric field. Here we use the first numeric-type field found for demonstration. (Change `numeric_field_id` and `group_field_id` as needed.)
2. Filter the DataFrame for records where the numeric field exceeds a threshold.
3. Normalize the filtered field using z-score normalization.
4. Optionally, group by another field.


In [ ]:
# Select a record set that has loaded records and numeric fields
import numpy as np

# Helper to pick a numeric field
selected_df = None
selected_rs = None
numeric_field_id = None
group_field_id = None

for rs in record_sets:
    rs_id = rs.id
    df = all_dataframes.get(rs_id)
    if df is not None and not df.empty:
        for field in rs.fields:
            # Look for a float/integer field
            if getattr(field, 'data_type', None) in ['Float', 'Number', 'Integer']:
                numeric_field_id = field.id
                selected_df = df
                selected_rs = rs
                break
    if numeric_field_id:
        break

if numeric_field_id:
    print(f"Selected record set: {selected_rs.name} (@id: {selected_rs.id})")
    print(f"Numeric field chosen: {numeric_field_id}")
    # Try to find a non-numeric field for grouping
    for field in selected_rs.fields:
        if getattr(field, 'data_type', None) not in ['Float', 'Number', 'Integer']:
            group_field_id = field.id
            break

    # Drop NA for numeric field and ensure numeric dtype
    df_num = pd.to_numeric(selected_df[numeric_field_id], errors='coerce')
    filtered_df = selected_df[df_num > 10].copy()
    filtered_df[numeric_field_id] = df_num[df_num > 10]

    print(f"Filtered records with {numeric_field_id} > 10 (found {filtered_df.shape[0]} rows):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std())

    print(f"\nNormalized '{numeric_field_id}' (z-score):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional: grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in the loaded dataset.")

## 5. Visualization

Let's plot the distribution of the chosen numeric field and, if available, a boxplot grouped by the chosen categorical field (if one exists).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(selected_df[numeric_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in selected_df.columns:
        plt.figure(figsize=(10, 5))
        # Make sure numeric column is numeric and drop NA
        plot_df = selected_df[[group_field_id, numeric_field_id]].copy()
        plot_df[numeric_field_id] = pd.to_numeric(plot_df[numeric_field_id], errors='coerce')
        plot_df = plot_df.dropna(subset=[numeric_field_id])
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=plot_df)
        plt.xticks(rotation=45)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.show()
else:
    print("No suitable data available for visualization.")

## 6. Conclusion

- This notebook demonstrated how to load and inspect a multi-record-set scientific dataset using `mlcroissant`.
- Record set and field `@id`s were used for all references, ensuring robust access across the schema.
- We performed basic data extraction and exploratory analysis of numeric fields, including filtering and normalization, and produced summary visualizations.

Further analyses, such as deeper statistical exploration, advanced visualizations, and machine learning, can be performed now that the data is loaded in a standard pandas format.